# einops-reduce — worked example 1: Sum over the batch axis to get a per-channel total

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-reduce`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import reduce

t.manual_seed(0)
np.random.seed(0)

## Concept

`einops.reduce` collapses any axis that appears on the left side of the pattern but not on the right. Using `'b c -> c'` with `'sum'` adds up all batch elements for each channel, producing a single per-channel accumulator. This is the einops equivalent of `x.sum(dim=0)` but with named axes that make the intent explicit.

## Worked solution

Input: `(B=4, C=3)` tensor of per-sample per-channel losses.

**Pattern:** `'b c -> c'` with `'sum'`. The `b` axis appears on the left but not the right — it gets reduced. The `c` axis survives.

**Computation:** for each channel `c`, sum over all `B=4` batch entries: `out[c] = sum_b x[b, c]`.

**Result shape:** `(C=3,)`.

This pattern is useful when you want the total loss per channel before dividing by batch size, or when accumulating gradients manually.

In [ ]:
import torch as t
from einops import reduce

t.manual_seed(99)
B, C = 5, 4
x = t.randn(B, C).abs()  # positive per-sample per-channel losses

print('Input shape:', x.shape)
print('Input values:\n', x)

channel_total = reduce(x, 'b c -> c', 'sum')
print('Per-channel sum shape:', channel_total.shape)  # (4,)
print('Per-channel sum:', channel_total)

# Verify against manual sum
assert t.allclose(channel_total, x.sum(dim=0))
print('Matches x.sum(dim=0):', True)